# 08 Multi-view clustering: artists + albums + tracks

In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

TARGET = "tracks"          # the view that is recommended and evaluated
N_USERS = None               # None = all users, e.g. 20_000 for a quick test

sys.path.append("..")          # cf_common.py, cure.py and neighbours.py are in the folder above
DATA_DIR = Path("../data")

from cure import cure
from cf_common import Data, load_data, evaluate_clustering, save_results, assign_to_nearest
from neighbours import true_neighbours, neighbour_recall

VIEWS = ["artists", "albums", "tracks"]
EXTRA_VIEWS = [view for view in VIEWS if view != TARGET]

# the same settings as 00_prepare_data and 02_cure
N_HIDDEN = 10
MIN_USERS_PER_ITEM = 20
EMBEDDING_DIM = 8
N_EVAL_USERS = 10_000
SEED = 579
K_VALUES = [10, 20, 40, 80]
SAMPLE_SIZE = 10_000
N_REPRESENTATIVES = 8
SHRINK = 0.8
MIN_CLUSTER_SIZE = 10

print(f"target view: {TARGET}; extra views: {', '.join(EXTRA_VIEWS)}")

target view: tracks; extra views: artists, albums


## Step 1: tracks and the train/test split

Same train/test split as the other notebooks

In [2]:
TITLE_COLUMN = {"artists": None, "albums": "album_name", "tracks": "track_name"}


def load_view(view):
    """Loads one view's CSV and cleans it like 00_prepare_data. The column 'item' numbers the items."""
    title = TITLE_COLUMN[view]
    columns = ["user_id", "artist_name", "playcount"] + ([title] if title else [])
    dtypes = {"user_id": "int32", "playcount": "int32", "artist_name": "category"}
    if title:
        dtypes[title] = "category"
    df = pd.read_csv(DATA_DIR / f"user_top_{view}.csv", usecols=columns, dtype=dtypes,
                     nrows=N_USERS * 50 if N_USERS else None, keep_default_na=False, na_values=[""])
    df = df.dropna(subset=[c for c in columns if c.endswith("_name")])
    df = df[df["artist_name"] != "[unknown]"]
    if title:
        df["item"] = df.groupby([title, "artist_name"], observed=True).ngroup().astype("int32")
    else:
        df["item"] = df["artist_name"].cat.remove_unused_categories().cat.codes.astype("int32")
    return df


def build_matrix(rows, n_items):
    """User x item matrix with log(1 + playcount), only items that at least 20 users have.
    Also returns item id -> column (-1 = not in the matrix)."""
    users_per_item = np.bincount(rows["item"], minlength=n_items)
    vocabulary = np.flatnonzero(users_per_item >= MIN_USERS_PER_ITEM)
    column = np.full(n_items, -1)
    column[vocabulary] = np.arange(len(vocabulary))
    rows = rows[column[rows["item"]] >= 0]
    matrix = sp.csr_matrix(
        (np.log1p(rows["playcount"].to_numpy()).astype(np.float32),
         (rows["user"].to_numpy(), column[rows["item"]])),
        shape=(n_users, len(vocabulary)))
    return matrix, column


start = time.perf_counter()
target = load_view(TARGET)
user_ids = target["user_id"].astype("category").cat.categories       # user number = position in this list
target["user"] = target["user_id"].astype("category").cat.codes.astype("int32")
n_users = len(user_ids)

# hide 10 random items per user
target = target.sample(frac=1, random_state=SEED)
position = target.groupby("user").cumcount()
n_items_of_user = target.groupby("user")["item"].transform("size")
target["hidden"] = (position < N_HIDDEN) & (n_items_of_user >= 2 * N_HIDDEN)

target_matrix, target_column = build_matrix(target[~target["hidden"]], target["item"].max() + 1)
svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=SEED)
target_embedding = normalize(svd.fit_transform(normalize(target_matrix)))

# the 10,000 test users and their hidden items
rng = np.random.default_rng(SEED)
users_with_hidden = np.unique(target.loc[target["hidden"], "user"])
eval_users = np.sort(rng.choice(users_with_hidden, size=min(N_EVAL_USERS, len(users_with_hidden)), replace=False))
hidden = target[target["hidden"] & target["user"].isin(eval_users)].sort_values("user")
hidden_items = target_column[hidden["item"]].reshape(len(eval_users), N_HIDDEN)
data = Data(target_matrix, target_embedding, eval_users, hidden_items)

print(f"{TARGET}: {n_users:,} users, matrix {target_matrix.shape[1]:,} items ({time.perf_counter() - start:.0f}s)")

tracks: 476,451 users, matrix 120,650 items (75s)


## Step 2: artists and albums, with the plays of the hidden tracks subtracted

In [ ]:

hidden_rows = target.loc[target["hidden"], ["user", "artist_name", "playcount"]]
hidden_artist_names = pd.Index(hidden_rows["artist_name"].cat.categories.astype(str))


def artist_numbers(df):
    """Number of each row's artist in hidden_artist_names (-1 = not the artist of any hidden item)."""
    number_of_category = hidden_artist_names.get_indexer(df["artist_name"].cat.categories.astype(str))
    return number_of_category[df["artist_name"].cat.codes.to_numpy()]


# plays of each user's hidden tracks, summed per (user, artist)
keys = hidden_rows["user"].to_numpy().astype(np.int64) * len(hidden_artist_names) + artist_numbers(hidden_rows)
hidden_keys, position = np.unique(keys, return_inverse=True)
hidden_plays = np.bincount(position, weights=hidden_rows["playcount"].to_numpy()).astype(np.int64)


def plays_to_subtract(df):
    """For every row: the plays of this user's hidden tracks by the row's artist (0 if there are none)."""
    number = artist_numbers(df)
    keys = df["user"].to_numpy().astype(np.int64) * len(hidden_artist_names) + number
    i = np.minimum(np.searchsorted(hidden_keys, keys), len(hidden_keys) - 1)
    found = (number >= 0) & (hidden_keys[i] == keys)
    return np.where(found, hidden_plays[i], 0)


view_matrices = {TARGET: target_matrix}
for view in EXTRA_VIEWS:
    start = time.perf_counter()
    df = load_view(view)
    df["user"] = pd.Categorical(df["user_id"], categories=user_ids).codes    # the target's user numbers (-1 = unknown)
    df = df[df["user"] >= 0]
    subtract = plays_to_subtract(df)
    playcount = df["playcount"].to_numpy() - subtract
    kept = playcount > 0
    view_matrices[view], _ = build_matrix(df[kept].assign(playcount=playcount[kept]), df["item"].max() + 1)
    matrix = view_matrices[view]
    print(f"{view:<8}: {len(df):,} rows, {(subtract > 0).sum():,} lowered ({100 * (subtract > 0).mean():.1f}%), "
          f"{(~kept).sum():,} removed ({100 * (~kept).mean():.1f}%), {100 * np.minimum(subtract, df['playcount']).sum() / df['playcount'].sum():.1f}% "
          f"of the plays subtracted, matrix {matrix.shape[1]:,} items, "
          f"{(matrix.getnnz(axis=1) == 0).sum():,} users with an empty vector ({time.perf_counter() - start:.0f}s)")
    del df

del target          # the raw rows are not needed any more

artists : 23,814,295 rows, 2,784,426 lowered (11.7%), 60,452 removed (0.3%), 3.4% of the plays subtracted, matrix 50,911 items, 16 users with an empty vector (39s)


albums  : 23,815,046 rows, 6,638,884 lowered (27.9%), 1,559,700 removed (6.5%), 20.9% of the plays subtracted, matrix 92,696 items, 359 users with an empty vector (77s)


## Step 3 and 4: combine the views

In [4]:
def embed(matrix):
    svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=SEED)
    return normalize(svd.fit_transform(normalize(matrix))), 100 * svd.explained_variance_ratio_.sum()


view_embeddings = {TARGET: target_embedding}
for view in EXTRA_VIEWS:
    view_embeddings[view], variance = embed(view_matrices[view])
    print(f"{view:<8} 8-dim embedding: {variance:.1f}% of the variance")

concatenated = normalize(np.hstack([view_embeddings[view] for view in VIEWS]))
joint, variance = embed(sp.hstack([normalize(view_matrices[view]) for view in VIEWS]).tocsr())
print(f"joint SVD of all views: {variance:.1f}% of the variance")

EMBEDDINGS = {"concatenated (24-dim)": concatenated, "joint SVD (8-dim)": joint}

artists  8-dim embedding: 10.1% of the variance


albums   8-dim embedding: 4.0% of the variance


joint SVD of all views: 4.6% of the variance


## Step 5: CURE and k-means on the multi-view embeddings
Same settings and CURE sample as `02_cure` and `03_kmeans`. Neighbour recall = share of each test user's 50 true
nearest neighbours (on tracks) that are in the same cluster.

In [5]:
neighbours = true_neighbours(data, cache_file="cache/true_neighbours.npy" if N_USERS is None
                             else "cache/true_neighbours_test.npy")
sample = np.random.default_rng(579).choice(n_users, size=min(SAMPLE_SIZE, n_users), replace=False)
summaries, per_user = [], []
saved_labels = {}          # cluster labels, read by 09_view_clusters


def evaluate(method, k, labels, fit_seconds, embedding_name):
    saved_labels[f"{method.split()[-1]}_{embedding_name.split()[0]}_k{k}"] = labels
    recall = 100 * neighbour_recall(data, labels, neighbours)
    summary, users = evaluate_clustering(data, method, k, labels, fit_seconds,
                                         extra={"embedding": embedding_name, "neighbour recall %": recall})
    summaries.append(summary)
    per_user.append(users)


for name, points in EMBEDDINGS.items():
    print(f"--- {name}")
    start = time.perf_counter()
    representatives = cure(points[sample], K_VALUES, N_REPRESENTATIVES, SHRINK, MIN_CLUSTER_SIZE)
    cure_seconds = time.perf_counter() - start
    for k in K_VALUES:
        start = time.perf_counter()
        rep_points, rep_owner = representatives[k]
        labels = rep_owner[assign_to_nearest(points, rep_points)]
        evaluate("multiview cure", k, labels, cure_seconds + time.perf_counter() - start, name)

    for k in K_VALUES:
        start = time.perf_counter()
        labels = KMeans(n_clusters=k, n_init=3, random_state=579).fit_predict(points)
        evaluate("multiview kmeans", k, labels, time.perf_counter() - start, name)

np.savez("cache/multiview_labels.npz", **saved_labels)
results = save_results("multiview", summaries, per_user, folder="results/analyses")

--- concatenated (24-dim)


multiview cure   k=10   P@10 = 0.1356   clustering   61.1s   CF   75.3s


multiview cure   k=20   P@10 = 0.1319   clustering   61.2s   CF   58.8s


multiview cure   k=40   P@10 = 0.1316   clustering   61.1s   CF   41.7s


multiview cure   k=80   P@10 = 0.1231   clustering   61.3s   CF   23.7s


multiview kmeans k=10   P@10 = 0.1233   clustering    1.6s   CF   26.3s


... (8 outputs left out)

multiview kmeans k=20   P@10 = 0.1185   clustering    5.6s   CF   22.5s


multiview kmeans k=40   P@10 = 0.1114   clustering   11.4s   CF   20.6s


multiview kmeans k=80   P@10 = 0.1057   clustering   19.3s   CF   19.6s


saved results\analyses\multiview.csv and results\analyses\multiview_per_user.csv


## Step 6: compare with single-view
The single-view results come from `results/` (01, 02, 03). For every multi-view run the table shows the
single-view result of the same method at the same neighbour pool (interpolated between its k values; empty if
the pool is outside the single-view range). Positive difference = multi-view is better.

In [6]:
main = pd.concat([pd.read_csv(f) for f in sorted(Path("results").glob("*.csv")) if not f.stem.endswith("_per_user")],
                 ignore_index=True)
no_clustering = main.loc[main["method"] == "no_clustering", "P@10"].iloc[0]


def single_view_at(method, pool):
    """P@10 of a single-view method at this neighbour pool (log-linear interpolation between its k values)."""
    r = main[main["method"] == method].sort_values("avg neighbour pool")
    if len(r) == 0 or not r["avg neighbour pool"].min() <= pool <= r["avg neighbour pool"].max():
        return np.nan
    return np.interp(np.log(pool), np.log(r["avg neighbour pool"]), r["P@10"])


table = results[["method", "embedding", "k", "clusters", "largest cluster %", "avg neighbour pool",
                 "neighbour recall %", "P@10", "clustering s"]].copy()
table["single-view same method, same pool"] = [single_view_at(m.replace("multiview ", ""), p)
                                               for m, p in zip(table["method"], table["avg neighbour pool"])]
table["difference"] = table["P@10"] - table["single-view same method, same pool"]
print(f"no clustering ({TARGET}): P@10 = {no_clustering:.4f}")
table.round(4)

no clustering (tracks): P@10 = 0.1356


,method,embedding,k,clusters,largest cluster %,avg neighbour pool,neighbour recall %,P@10,clustering s,"single-view same method, same pool",difference
0,multiview cure,concatenated (24-dim),10,1,100.0000,476451.0000,100.0000,0.1356,61.1219,NaN,NaN
1,multiview cure,concatenated (24-dim),20,3,80.2551,316412.3015,80.6762,0.1319,61.1674,NaN,NaN
2,multiview cure,concatenated (24-dim),40,3,72.5443,268425.9311,77.4552,0.1316,61.1432,NaN,NaN
3,multiview cure,concatenated (24-dim),80,12,34.3886,80887.8938,53.8810,0.1231,61.2721,0.1226,0.0006
4,multiview kmeans,concatenated (24-dim),10,10,14.7652,51007.1890,51.7900,0.1233,1.6136,0.1207,0.0025
5,multiview kmeans,concatenated (24-dim),20,20,7.2964,25414.3508,40.4578,0.1178,6.1209,0.1147,0.0031
6,multiview kmeans,concatenated (24-dim),40,40,4.2632,12919.7746,28.9142,0.1119,20.4789,0.1075,0.0044
7,multiview kmeans,concatenated (24-dim),80,80,2.6643,6693.6917,19.3258,0.1042,11.4254,NaN,NaN
8,multiview cure,joint SVD (8-dim),10,4,58.8659,199718.8036,74.7264,0.1307,30.8856,NaN,NaN
9,multiview cure,joint SVD (8-dim),20,9,32.3473,85446.0291,55.2770,0.1260,31.0090,0.1234,0.0026
